# पाठ ०१ - AI एजेन्टहरू परिचय

**शुरुआतकर्ताहरूका लागि AI एजेन्टहरू** को पाठक्रमको पहिलो पाठमा स्वागत छ!

एउटा **AI एजेन्ट** यस्तो कार्यक्रम हो जुन ठूलो भाषा मोडेल (LLM) लाई यसको तर्क इञ्जिनको रूपमा प्रयोग गर्छ र वास्तविक संसारमा *कार्यहरू* गर्न सक्छ — API कल गर्ने, डाटाबेस जाँच्ने, वा कोड चलाउने — प्रयोगकर्ताको तर्फबाट एक लक्ष्य पूरा गर्न।

यस नोटबुकमा तपाईँले आफ्नो पहिलो एजेन्ट निर्माण गर्नुहुनेछ: एक **यात्रा एजेन्ट** जुन बिदाको गन्तव्य सल्लाह दिन्छ। यस क्रममा तपाईँले सिक्नुहुनेछ:

१. **Microsoft Agent Framework** प्रयोग गरेर Microsoft Foundry Agent Service सँग जडान गर्ने।
२. एजेन्टलाई एउटा **उपकरण** दिने — एउटा साधारण Python फङ्क्शन जुन यो कल गर्न सक्छ।
३. एजेन्ट चलाउने र यसको प्रतिक्रिया निरीक्षण गर्ने।
४. एजेन्टको प्रतिक्रियालाई टोकन-द्वारा-टोकन स्ट्रिम गर्ने।


## सेटअप

यो नोटबुक चलाउनुअघि, सुनिश्चित गर्नुस् कि तपाईंसँग:

1. **डिप्लोय गरिएको च्याट मोडल सहितको माइक्रोसफ्ट फाउन्ड्री प्रोजेक्ट** (जस्तै `gpt-5-mini`) छ।
2. **एज्योर CLI मार्फत लगइन गरिएको छ** — तपाईंको टर्मिनलमा `az login` चलाउनुहोस्।
3. **आवश्यक वातावरण चरहरू सेट गर्नुभएको छ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तपाईंको माइक्रोसफ्ट फाउन्ड्री प्रोजेक्टको अन्त्यबिन्दु।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईंले डिप्लोय गरेको मोडलको नाम।

तलको सेलले तपाईंलाई आवश्यक पायथन प्याकेजहरू इन्स्टल गर्दछ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## तपाईंको पहिलो एजेन्ट बनाउँदै

एउटा एजेन्टलाई दुई कुरा चाहिन्छ:

- **निर्देशनहरू** जसले यसलाई *को हो* र *कसरी व्यवहार गर्ने* बताउँछन् (एक प्रणाली प्रॉम्प्ट)।
- **उपकरणहरू** — Python कार्यहरू जुन `@tool` सँग सजिएका हुन्छन् र एजेन्टले जानकारी प्राप्त गर्न वा क्रियाहरू गर्न कल गर्न सक्छ।

तल हामी एउटा साधारण उपकरण परिभाषित गर्छौं जुन लोकप्रिय छुट्टी गन्तव्यहरूको सूची फर्काउँछ। प्रयोगकर्ताले यात्रा सिफारिसहरूको लागि सोधेमा एजेन्टले यो उपकरण प्रयोग गर्नेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रिमिङ प्रतिक्रियाहरू

अझ अन्तरक्रियात्मक अनुभवको लागि तपाईं एजेन्टको प्रतिक्रियालाई **स्ट्रिम** गर्न सक्नुहुन्छ। पूर्ण जवाफको पर्खाइ नगरी, एजेन्टले उत्पन्न हुँदा पाठका भागहरू दिन्छ। यो विशेष गरी च्याट इन्टरफेसहरूमा उपयोगी छ जहाँ तपाईं वास्तविक समयमा आउटपुट देखाउन चाहनुहुन्छ।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

यस पाठमा तपाईंले सिक्नुभयो कसरी:

- **प्रदायक बनाउने** जसले Microsoft Foundry Agent Service सँग `FoundryChatClient` मार्फत जडान गर्छ।
- **उपकरण परिभाषित गर्ने** `@tool` डेकोरेटर प्रयोग गरेर ताकि एजेन्टले तपाईंका Python फङ्सनहरू कल गर्न सकोस्।
- **एजेन्ट चलाउने** प्रयोगकर्ता सन्देशसहित र यसको प्रतिक्रिया प्रिन्ट गर्ने।
- **प्रतिक्रियाहरू स्ट्रिम गर्ने** वास्तविक-समय आउटपुटका लागि।

अर्को पाठमा हामी एजेन्टिक फ्रेमवर्कहरूलाई अझ गहिराइमा अन्वेषण गर्नेछौं र एजेन्टहरूलाई थप शक्तिशाली उपकरणहरू र बहु-चरण तर्क क्षमताहरू कसरी दिने भन्ने सिक्नेछौं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
